# TensorFlow: Build multi-input model using siamese network architecture

In [ ]:
import os
import random
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "1"

import tensorflow as tf
from tensorflow.keras import backend as K
from tensorflow.keras import ops
from tensorflow.keras.datasets import fashion_mnist

print("TF Version: ", tf.__version__)
print("TF Eager mode: ", tf.executing_eagerly())
print("TF GPU is", "available" if tf.config.list_physical_devices("GPU") else "not available")

# Prepare dataset

In [ ]:
MNIST_CLASSES = 10

In [ ]:
def create_pairs(x, indices):
    pairs = []
    labels = []
    n = min([len(indices[d]) for d in range(MNIST_CLASSES)]) - 1
    for d in range(MNIST_CLASSES):
        for i in range(n):
            z1, z2 = indices[d][i], indices[d][i + 1]
            pairs += [[x[z1], x[z2]]]
            inc = random.randrange(1, MNIST_CLASSES)
            dn = (d + inc) % MNIST_CLASSES
            z1, z2 = indices[d][i], indices[dn][i]
            pairs += [[x[z1], x[z2]]]
            labels += [1, 0]
    return np.array(pairs), np.array(labels)

In [ ]:
def create_pairs_on_set(images, labels):
    indices = [np.where(labels == i)[0] for i in range(MNIST_CLASSES)]
    pairs, y = create_pairs(images, indices)
    y = y.astype("float32")
    return pairs, y

In [ ]:
def show_pair(images, label):
    plt.figure()
    plt.subplot(211)
    plt.axis("off")
    plt.title("Similar" if label == 1.0 else "Not Similar")
    plt.imshow(images[0])
    plt.subplot(212)
    plt.axis("off")
    plt.imshow(images[1])
    plt.grid(False)
    plt.show()

In [ ]:
(tr_images, tr_labels), (ts_images, ts_labels) = fashion_mnist.load_data()

In [ ]:
# Normalize images
tr_images = tr_images.astype(np.float32) / 255.0
ts_images = ts_images.astype(np.float32) / 255.0

In [ ]:
# Create pairs of images (image1, image2) and particular label (1 - similar, 0 - dissimilar)
tr_x, tr_y = create_pairs_on_set(tr_images, tr_labels)
ts_x, ts_y = create_pairs_on_set(ts_images, ts_labels)

In [ ]:
show_pair(tr_x[10], tr_y[10])

In [ ]:
show_pair(tr_x[13], tr_y[13])

# Build Model

__Create model__

In [ ]:
# Builds base deep neural network
def get_base_network():
    input = tf.keras.Input(shape=(28,28), name="input")
    x = tf.keras.layers.Flatten()(input)
    x = tf.keras.layers.Dense(units=128, activation=tf.nn.relu)(x)
    x = tf.keras.layers.Dropout(0.1)(x)
    x = tf.keras.layers.Dense(units=128, activation=tf.nn.relu)(x)
    x = tf.keras.layers.Dropout(0.1)(x)
    # Output embedded vector for input image
    output = tf.keras.layers.Dense(units=128, activation=tf.nn.relu)(x)
    return tf.keras.Model(inputs=input, outputs=output)

In [ ]:
# Calculates Euclidean distance of two embedded vectors (image_1 vector + image_2 vector)
def euclidean_distance(vectors):
    y1, y2 = vectors
    # Avoid zero distances by substituting non-zero constant
    D = tf.maximum(y1 - y2, K.epsilon())
    sum_square = tf.reduce_sum(tf.square(D), axis=1, keepdims=True)
    return tf.sqrt(tf.maximum(sum_square, K.epsilon()))

In [ ]:
# Calculates a shape of outputs as one number for each pair in a batch
def euclidean_shape(shapes):
    shape1, shape2 = shapes
    return shape1[0], 1

In [ ]:
base = get_base_network()

# image_1 input
input1 = tf.keras.Input(shape=(28,28,))
# image_2 input
input2 = tf.keras.Input(shape=(28,28,))

# image_1 embedded vector
output1 = base(input1)
# image_2 embedded vector
output2 = base(input2)

# outputs is a distance between two embedded vectors
output = tf.keras.layers.Lambda(
    euclidean_distance,
    name="output",
    output_shape=euclidean_shape)([output1, output2])

model = tf.keras.Model(inputs=[input1, input2], outputs=output)

In [ ]:
# Create contrastive loss function by subclass the Loss class
class ContrastiveLoss(tf.keras.losses.Loss):
    margin = 1.0

    def __init__(self, margin, **kwargs):
        super().__init__(**kwargs)
        self.margin = margin

    def call(self, y_true, y_pred):
        # 1: Make sure the underlying type is float
        y_true = tf.cast(y_true, tf.float32)
        # 2: Make sure the shapes of y_true and y_pred are the same
        y_true = tf.reshape(y_true, tf.shape(y_pred))
        pos_loss = tf.square(y_pred) # Dw^2
        neg_loss = tf.square(tf.maximum(self.margin - y_pred, 0)) # max(0, m - Dw)^2
        # 3: Do not use tf.reduce_mean as it is already done in a base class
        return y_true * pos_loss + (1 - y_true) * neg_loss # Y*Dw^2 + (Y-1)*max(0, m - Dw)^2

    def get_config(self):
        config = super().get_config()
        config.update({"margin": self.margin})
        return config

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.RMSprop(),
    loss=ContrastiveLoss(margin=1.0))

__Train model__

In [ ]:
history = model.fit(
    x=[tr_x[:,0], tr_x[:,1]],
    y=tr_y,
    epochs=20,
    batch_size=64,
    validation_data=([ts_x[:,0], ts_x[:,1]], ts_y))

__Evaluate model__

In [ ]:
def compute_accuracy(y_true, y_pred):
    pred = y_pred.ravel() < 0.5
    return np.mean(pred == y_true)

In [ ]:
loss = model.evaluate(x=[ts_x[:,0], ts_x[:,1]], y=ts_y)
print(f"Total loss: {loss:.3}")

In [ ]:
y_pred_tr = model.predict(x=[tr_x[:,0], tr_x[:,1]])
tr_accuracy = compute_accuracy(tr_y, y_pred_tr)
print(f"Train accuracy: {tr_accuracy}")

y_pred_ts = model.predict(x=[ts_x[:,0], ts_x[:,1]])
ts_accuracy = compute_accuracy(ts_y, y_pred_ts)
print(f"Test accuracy: {ts_accuracy}")

In [ ]:
def plot_metrics(history, metric, title, ylim=0.25):
    plt.title(title)
    plt.ylim(0, ylim)
    plt.plot(history[metric], color="blue", label=metric)
    plt.plot(history["val_" + metric], color="green", label="val_" + metric)

In [ ]:
plot_metrics(history.history, metric="loss", title="Loss")